# 00 Firm Selection

This notebook identifies three French environmental, sustainability, or carbon-related organizations with published BEGES reports on the ADEME platform.

The selected firms will be used for a comparative GHG inventory and reporting-quality analysis in the `bilan-critique` project.

This notebook uses the ADEME open-data export and DuckDB so that the firm selection process is reproducible, transparent, and based on SQL queries rather than manual searching alone.

In [33]:
import duckdb
import pandas as pd
from pathlib import Path

conn = duckdb.connect()

csv_path = Path("../data/raw/export-opendata-inventories-03-05-2026.csv")

print("Libraries loaded successfully")
print(csv_path.exists())

Libraries loaded successfully
True


## Load the ADEME BEGES export

This block loads the raw ADEME BEGES export into a pandas dataframe.

The ADEME file uses semicolons as column separators, which is common in French open-data CSV files. The dataset is loaded without modification so that the original structure is preserved.

The shape of the dataframe is printed to confirm the number of rows and columns. The first rows are displayed to verify that the file loaded correctly.

In [34]:
df = pd.read_csv(csv_path, sep=";", low_memory=False)

print(df.shape)

df.head()

(11138, 104)


,Id,"Méthode BEGES (V4,V5)",Date de publication,Type de structure,Type de collectivité,Raison sociale,SIREN principal,APE(NAF) associé,Libellé,Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan,...,Incertitudes,Sources,Recalcul,Siret,Comparaison avec le précédent bilan,Lien URL vers le rapport complet du BEGES,Responsable du suivi,Fonction,Téléphone,Courriel
0,503c7fae-4344-4957-b7bc-d891c186d647,v5,01/01/2025,Entreprise,NaN,SERIDERM FRANCE,843307646,4645Z,Commerce de gros (commerce interentreprises) d...,Entre 20 et 49,...,NaN,Les principales sources de facteurs d&#039;émi...,NaN,84330764600029,NaN,NaN,[Masqué],[Masqué],[Masqué],[Masqué]
1,9381ca45-b1cd-11ed-8fce-005056b7acd1,v4,01/02/2017,Établissement public,NaN,CENTRE HOSPITALIER DE LA COTE FLEURIE,200017986,8610Z,Activités hospitalières,Entre 500 et 999,...,l&#039;incertitude sur le bilan global est de ...,NaN,NaN,NaN,NaN,NaN,[Masqué],[Masqué],[Masqué],[Masqué]
2,9381cdd9-b1cd-11ed-8fce-005056b7acd1,v4,01/02/2017,Entreprise,NaN,AREVA NP SAS,428764500,2530Z,"Fabrication de générateurs de vapeur, à l'exce...",Entre 5 000 et 9 999,...,NaN,NaN,NaN,NaN,NaN,NaN,Maxime PILLIE,Responsable Développement Durable et RSE,NaN,maxime.pillie@areva.com
3,9381cfcc-b1cd-11ed-8fce-005056b7acd1,v4,01/02/2017,Entreprise,NaN,GROUPE SECOB RENNES,424936656,6920Z,Activités comptables,Entre 50 et 99,...,NaN,NaN,NaN,NaN,NaN,NaN,[Masqué],[Masqué],[Masqué],[Masqué]
4,937cbcaf-b1cd-11ed-8fce-005056b7acd1,v4,01/02/2017,Entreprise,NaN,GROUPAMA D'OC,391851557,6512Z,Autres assurances,Entre 1 000 et 1 999,...,\n\t\n\t\t\n\t\t\t\n\t\t\tPoste d’émissions\n\...,NaN,NaN,\r\n\t\r\n\t\t\r\n\t\t\tENTREPRISE\r\n\t\t\tSI...,NaN,NaN,[Masqué],[Masqué],[Masqué],[Masqué]


In [35]:
df.shape

(11138, 104)

## Inspect available columns

This block displays the column names in the ADEME export.

Before writing SQL queries, the exact column names must be inspected because the dataset uses French administrative labels. These names will be used to filter organizations, reporting years, emissions scopes, and business sectors.

In [36]:
df.columns.tolist()

['Id',
 'Méthode BEGES (V4,V5)',
 'Date de publication',
 'Type de structure',
 'Type de collectivité',
 'Raison sociale',
 'SIREN principal',
 'APE(NAF) associé',
 'Libellé',
 "Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan",
 'Population',
 'Région',
 'Code département',
 'Département',
 'Structure obligée',
 'SIREN des entités consolidées',
 'Code NAF des entités consolidées',
 'Départements des entités consolidées',
 'Statut des entités consolidées',
 'Mode de consolidation',
 'Année de reporting',
 'Assujetti DPEF/PCAET ?',
 'Lien DPEF/PCAET',
 'Assujetti à la CSRD',
 'Périmètre national décrit dans CSRD',
 'Lien CSRD',
 'Pages CSRD correspondantes',
 "Aide diag décarbon'action",
 "Seuil d'importance retenu (%)",
 "Niveau d'influence",
 'Importance stratégique et vulnérabilités',
 'Lignes directrices spécifiques au secteur',
 'Sous-traitance',
 'Engagement du personnel',
 "Justification des postes d'émissions indirec

## Register the ADEME dataframe as a SQL table

This block registers the raw ADEME dataframe as a DuckDB table named `beges`.

This allows the notebook to query the dataset using SQL. SQL is useful here because the first project task is not visualization, but structured filtering: identifying organizations by type, sector, reporting year, CSRD status, and emissions completeness.

In [37]:
conn.register("beges", df)

conn.sql("""
SELECT 
    "Type de structure",
    COUNT(*) AS number_of_reports
FROM beges
GROUP BY "Type de structure"
ORDER BY number_of_reports DESC
""").df()

,Type de structure,number_of_reports
0,Entreprise,7820
1,Établissement public,1876
2,Collectivité territoriale (dont EPCI),738
3,État,392
4,Association,312


## Search the full ADEME dataset for candidate organizations

This block searches the complete ADEME dataset for organizations whose names suggest a connection to carbon, sustainability, climate consulting, environmental consulting, or technical environmental services.

At this stage, no structure type is excluded. This avoids accidentally removing relevant organizations that may be registered as associations, public entities, or another legal category.

In [38]:
candidate_search = conn.sql("""
SELECT
    "Id",
    "Raison sociale",
    "Type de structure",
    "Libellé",
    "Année de reporting",
    "Assujetti à la CSRD",
    "Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan" AS staff_count,
    "Emissions publication P1.1",
    "Emissions publication P2.1",
    "Emissions publication P3.1",
    "Emissions publication P4.1",
    "Emissions publication P5.1",
    "Lien URL vers le rapport complet du BEGES"
FROM beges
WHERE UPPER("Raison sociale") LIKE '%CARBONE%'
   OR UPPER("Raison sociale") LIKE '%CLIMAT%'
   OR UPPER("Raison sociale") LIKE '%ENVIRON%'
   OR UPPER("Raison sociale") LIKE '%GREEN%'
   OR UPPER("Raison sociale") LIKE '%ECO%'
   OR UPPER("Raison sociale") LIKE '%DURABLE%'
   OR UPPER("Raison sociale") LIKE '%SUSTAIN%'
ORDER BY "Année de reporting" DESC
""").df()

candidate_search.shape

(268, 13)

## Review keyword search results

This block displays the candidate organizations found through the initial keyword search.

The goal is to inspect whether the search is finding relevant carbon, sustainability, climate, and environmental organizations before applying stricter selection criteria.

In [39]:
candidate_search[
    [
        "Id",
        "Raison sociale",
        "Type de structure",
        "Libellé",
        "Année de reporting",
        "staff_count",
        "Assujetti à la CSRD"
    ]
].head(50)

,Id,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,Assujetti à la CSRD
0,a5f95a32-b811-4799-9818-f513dab0af38,TECHNI DECO S.A,Entreprise,Décolletage,2025,22,non
1,4ee63981-b1d8-47c1-92e5-69afe64bff85,ECOLE NATIONALE SUPERIEURE D'ARCHITECTURE ET D...,Établissement public,Enseignement supérieur,2025,150,NaN
2,b4d841bd-4b7a-477a-a3d9-f4cd7a12ec57,ECOLE NATIONALE SUPERIEURE DE MECANIQUE ET DES...,Établissement public,Enseignement supérieur,2025,147,NaN
3,ef76b5b1-82e7-47f4-a483-a52772e0e436,SCE DEPARTEMENTAL INCENDIE ET SECOURS,Établissement public,Services du feu et de secours,2024,4159,NaN
4,f36d813e-ccff-4147-94d4-8ab88799e206,SCE DEPARTEMENTAL INCENDIE ET SECOURS,Établissement public,"Administration publique (tutelle) de la santé,...",2024,1352,NaN
5,ddcb866a-51d5-4454-ac79-03d46719d0c8,"Direction régionale de l'environnement, de l'a...",État,Administration publique (tutelle) des activité...,2024,700,NaN
6,95a9d2c8-2be5-4d42-a5c0-4950a2a57ce8,SCOP EcoZimut,Entreprise,"Ingénierie, études techniques",2024,16,NaN
7,f166a771-2254-4a82-9b2c-034767471655,INSTITUT ENSEIGNEMENT SUPERIEUR ET RECHERCHE E...,Établissement public,Enseignement supérieur,2024,652,NaN
8,955a190b-6400-4b61-8566-97eb2ef1a8d1,ASSOCIATION ESTP - GRANDE ECOLE D'INGENIEURS D...,Association,Enseignement supérieur,2024,246,non
9,d70a0f1c-b69c-4600-994b-80d1c903b8b5,ECOCERT SA,Entreprise,"Analyses, essais et inspections techniques",2024,778,NaN


## Extract unique candidate organization names

This query returns unique organization names matching carbon, climate, sustainability, green, eco, or environmental keywords.

The purpose is to identify possible firms before choosing the final three BEGES reports for detailed analysis.

In [40]:
candidate_names = conn.sql("""
SELECT DISTINCT
    "Raison sociale",
    "Type de structure",
    "Libellé"
FROM beges
WHERE UPPER("Raison sociale") LIKE '%CARBONE%'
   OR UPPER("Raison sociale") LIKE '%CLIMAT%'
   OR UPPER("Raison sociale") LIKE '%ENVIRON%'
   OR UPPER("Raison sociale") LIKE '%GREEN%'
   OR UPPER("Raison sociale") LIKE '%ECO%'
   OR UPPER("Raison sociale") LIKE '%DURABLE%'
   OR UPPER("Raison sociale") LIKE '%SUSTAIN%'
ORDER BY "Raison sociale"
""").df()

candidate_names

,Raison sociale,Type de structure,Libellé
0,SOCIETE INSTALLATIONS TELECOMS COURANTS FORTS,Entreprise,Travaux d'installation électrique dans tous lo...
1,AC ENVIRONNEMENT,Entreprise,"Analyses, essais et inspections techniques"
2,AGENCE DE DEVELOPPEMENT ECONOMIQUE D'OCCITANIE,Entreprise,Autres activités de soutien aux entreprises n....
3,AGENCE DE L ENVIRONNEMENT ET DE LA MAITRISE DE...,Établissement public,Administration publique (tutelle) des activité...
4,ALGECO,Entreprise,"Location et location-bail d'autres machines, é..."
5,AQUITAINE VERRE DECOR,Entreprise,Activités spécialisées de design
6,ASSOCIATION ESTP - GRANDE ECOLE D'INGENIEURS D...,Association,Enseignement supérieur
7,All4green,Entreprise,Commerce de gros (commerce interentreprises) a...
8,Artemisia Environnement,Entreprise,Commerce de gros (commerce interentreprises) d...
9,BOUYGUES TELECOM,Entreprise,Télécommunications sans fil


In [41]:
structured_candidates = conn.sql("""
SELECT
    "Id",
    "Raison sociale",
    "Type de structure",
    "APE(NAF) associé",
    "Libellé",
    "Année de reporting",
    "Assujetti à la CSRD",
    "Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan" AS staff_count,
    "Mode de consolidation",
    "Emissions publication P1.1",
    "Emissions publication P2.1",
    "Emissions publication P3.1",
    "Emissions publication P3.2",
    "Emissions publication P3.3",
    "Emissions publication P4.1",
    "Emissions publication P4.2",
    "Emissions publication P5.1",
    "Actions et moyens",
    "Objectif de réduction pour 2030",
    "Lien URL vers le rapport complet du BEGES"
FROM beges
WHERE "Année de reporting" >= 2022
  AND (
      UPPER("Libellé") LIKE '%CONSEIL%'
      OR UPPER("Libellé") LIKE '%INGÉNIERIE%'
      OR UPPER("Libellé") LIKE '%INGENIERIE%'
      OR UPPER("Libellé") LIKE '%ÉTUDES%'
      OR UPPER("Libellé") LIKE '%ETUDES%'
      OR UPPER("Libellé") LIKE '%ENVIRONNEMENT%'
      OR UPPER("Libellé") LIKE '%TECHNIQUE%'
  )
ORDER BY "Année de reporting" DESC, staff_count DESC
""").df()

structured_candidates.shape

(573, 20)

## Inspect activity labels in the structured candidate pool

This query groups the candidate pool by activity label.

The goal is to identify which NAF activity categories are actually present in the filtered dataset, so the final firm selection can be based on relevant business activities rather than company-name guessing.

In [42]:
conn.register("structured_candidates", structured_candidates)

conn.sql("""
SELECT
    "Libellé",
    COUNT(*) AS number_of_reports
FROM structured_candidates
GROUP BY "Libellé"
ORDER BY number_of_reports DESC
""").df()

,Libellé,number_of_reports
0,"Ingénierie, études techniques",183
1,Conseil en systèmes et logiciels informatiques,147
2,Conseil pour les affaires et autres conseils d...,119
3,"Activités spécialisées, scientifiques et techn...",27
4,Fabrication de pièces techniques à base de mat...,24
5,"Analyses, essais et inspections techniques",22
6,Fabrication d'instrumentation scientifique et ...,19
7,Études de marché et sondages,12
8,Conseil en relations publiques et communication,7
9,Fabrication d'autres textiles techniques et in...,6


## Filter to relevant consultancy and technical advisory activities

This block narrows the candidate pool to activity labels that fit the project scope: engineering studies, management or sustainability consulting, and technical analysis or inspection.

These categories are more relevant for comparing environmental, sustainability, carbon, and technical advisory organizations than unrelated manufacturing or education activities.

In [43]:
relevant_candidates = conn.sql("""
SELECT *
FROM structured_candidates
WHERE "Libellé" IN (
    'Ingénierie, études techniques',
    'Conseil pour les affaires et autres conseils de gestion',
    'Analyses, essais et inspections techniques',
    'Activités spécialisées, scientifiques et techniques diverses'
)
ORDER BY "Année de reporting" DESC, staff_count DESC
""").df()

relevant_candidates.shape

(351, 20)

## Rank candidates by Scope 3 reporting depth

This block estimates Scope 3 reporting depth by summing the available indirect emissions publication columns.

For this project, firms with visible indirect emissions are more useful because the analysis focuses on Scope 3 coverage, reporting completeness, and alignment with modern climate disclosure expectations.

In [44]:
ranked_candidates = conn.sql("""
SELECT
    "Id",
    "Raison sociale",
    "Type de structure",
    "Libellé",
    "Année de reporting",
    staff_count,
    "Assujetti à la CSRD",
    "Mode de consolidation",
    (
        COALESCE("Emissions publication P3.1", 0) +
        COALESCE("Emissions publication P3.2", 0) +
        COALESCE("Emissions publication P3.3", 0) +
        COALESCE("Emissions publication P4.1", 0) +
        COALESCE("Emissions publication P4.2", 0) +
        COALESCE("Emissions publication P5.1", 0)
    ) AS visible_indirect_emissions,
    "Actions et moyens",
    "Objectif de réduction pour 2030",
    "Lien URL vers le rapport complet du BEGES"
FROM relevant_candidates
ORDER BY 
    "Année de reporting" DESC,
    visible_indirect_emissions DESC
""").df()

ranked_candidates.head(30)

,Id,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,Assujetti à la CSRD,Mode de consolidation,visible_indirect_emissions,Actions et moyens,Objectif de réduction pour 2030,Lien URL vers le rapport complet du BEGES
0,40a6ae82-78e0-42f3-ac7e-d92f6df13857,NXO FRANCE,Entreprise,"Ingénierie, études techniques",2025,1196,non,Opérationnel,8.280800e+04,Principales actions de décarbonation prévues s...,NaN,NaN
1,3bedd9e7-4de6-4a0c-9d51-dcaef764f1bf,SAFEGE,Entreprise,"Ingénierie, études techniques",2025,817,non,Opérationnel,2.789021e+03,Déplacements:Adaptation des modes de déplaceme...,-20% vs 2024: soit une réduction de 1007 tCO2e...,NaN
2,fb193005-867d-424c-971d-e78a983d2024,AGENCE FRANCAISE D'EXPERTISE TECHNIQUE INTERNA...,Établissement public,"Activités spécialisées, scientifiques et techn...",2025,2475,NaN,Opérationnel,1.813670e+03,Le Bilan Carbone de 2025 représente l&#039;ann...,NaN,NaN
3,aa321581-bde7-485f-bfc5-cb96647f8a0a,CIRTES SRC,Entreprise,"Activités spécialisées, scientifiques et techn...",2025,21,non,Opérationnel,4.811882e+02,Sortir certains produits du stock et rénover é...,Améliorer la performance énergétique des bâtim...,NaN
4,e75952c3-75c9-4182-b03e-fda04c203f05,Sol Solution,Entreprise,"Ingénierie, études techniques",2025,90,non,Opérationnel,4.223000e+02,SOL SOLUTION a engagé une démarche structurée ...,-40% par rapport à notre année de référence 2023,La Méthode Bilan Carbone® de l'ADEME
5,3c5d011c-5cd0-42b3-80e9-b73c062b067a,Humans Matter,Entreprise,Conseil pour les affaires et autres conseils d...,2025,61,NaN,Opérationnel,8.740000e+01,Humans Matter a identifié plusieurs actions pr...,Oui → réduction alignée SBTi :\nScope 1 & 2 : ...,NaN
6,ab5287c5-895a-44e9-aac7-035a4c3ec015,AQUASSAY,Entreprise,"Ingénierie, études techniques",2025,24,non,Opérationnel,7.500000e+01,NaN,12%,NaN
7,174bf12e-6f86-4aa4-a38f-fbd140a2ea10,INGENERIA,Entreprise,Conseil pour les affaires et autres conseils d...,2025,4,non,Opérationnel,5.000000e+01,Mobilisation des prestataires Préparation et ...,-20%,NaN
8,c9beb36d-1ea7-4c0a-aeca-f142890fd434,TASMANE,Entreprise,Conseil pour les affaires et autres conseils d...,2025,63,non,Opérationnel,4.100000e+01,Les actions de réduction identifiées s’articul...,Tasmane se fixe un objectif de réduction progr...,NaN
9,19541c22-bfe0-4e46-84b1-f1a70eae3f98,Conseil Formation Méditerranée,Entreprise,Conseil pour les affaires et autres conseils d...,2025,11,non,Opérationnel,1.601000e+01,RAS,NaN,NaN


## Search for named sustainability, carbon, and environmental advisory firms

This block searches the candidate pool for organizations that are likely to be directly relevant to the project theme.

At this stage, the goal is not to rank by emissions volume. The goal is to identify firms whose business activity makes them interesting subjects for a meta-analysis of GHG reporting quality.

In [45]:
target_firms = conn.sql("""
SELECT
    "Id",
    "Raison sociale",
    "Type de structure",
    "Libellé",
    "Année de reporting",
    staff_count,
    "Assujetti à la CSRD",
    "Mode de consolidation",
    visible_indirect_emissions,
    "Actions et moyens",
    "Objectif de réduction pour 2030",
    "Lien URL vers le rapport complet du BEGES"
FROM ranked_candidates
WHERE UPPER("Raison sociale") LIKE '%CARBONE%'
   OR UPPER("Raison sociale") LIKE '%GREEN%'
   OR UPPER("Raison sociale") LIKE '%ECO%'
   OR UPPER("Raison sociale") LIKE '%CLIMAT%'
   OR UPPER("Raison sociale") LIKE '%ENVIRON%'
   OR UPPER("Raison sociale") LIKE '%SUSTAIN%'
   OR UPPER("Raison sociale") LIKE '%I CARE%'
   OR UPPER("Raison sociale") LIKE '%EKODEV%'
   OR UPPER("Raison sociale") LIKE '%UTOPIES%'
   OR UPPER("Raison sociale") LIKE '%ECOACT%'
   OR UPPER("Raison sociale") LIKE '%GREENFLEX%'
   OR UPPER("Raison sociale") LIKE '%ARTELIA%'
   OR UPPER("Raison sociale") LIKE '%EGIS%'
   OR UPPER("Raison sociale") LIKE '%SETEC%'
   OR UPPER("Raison sociale") LIKE '%ANTEA%'
   OR UPPER("Raison sociale") LIKE '%SOCOTEC%'
ORDER BY "Année de reporting" DESC, visible_indirect_emissions DESC
""").df()

target_firms

,Id,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,Assujetti à la CSRD,Mode de consolidation,visible_indirect_emissions,Actions et moyens,Objectif de réduction pour 2030,Lien URL vers le rapport complet du BEGES
0,95a9d2c8-2be5-4d42-a5c0-4950a2a57ce8,SCOP EcoZimut,Entreprise,"Ingénierie, études techniques",2024,16,NaN,Opérationnel,6915.00,"Projets et usagesLâcher les gaz, pour plus de ...",NaN,NaN
1,7a0b6f91-9d03-4851-a2cc-9f2ab53b21d5,CARSO - LABORATOIRE SANTE ENVIRONNEMENT HYGIEN...,Entreprise,"Analyses, essais et inspections techniques",2024,890,NaN,Opérationnel,6453.00,Le groupe CARSO investit pour réduire ses émis...,42% de réduction des émissions,NaN
2,7abf0530-3f2c-46e1-8386-497989ef799c,SOCOTEC,Entreprise,"Analyses, essais et inspections techniques",2024,6075,oui,Opérationnel,4090.87,"Depuis 2022, SOCOTEC mets en œuvre un plan d’a...",42% de réduction sur le scope 1 et 2,NaN
3,d70a0f1c-b69c-4600-994b-80d1c903b8b5,ECOCERT SA,Entreprise,"Analyses, essais et inspections techniques",2024,778,NaN,Opérationnel,1242.70,La gouvernance a été renforcée avec l’évolutio...,NaN,NaN
4,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN
5,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN
6,5e1dcbd9-1848-4e05-a510-ef28436ed9ea,SAS OBJECTIF ZERO CARBONE CONSULTING (CARBONO),Entreprise,"Activités spécialisées, scientifiques et techn...",2024,6,NaN,Opérationnel,9.08,Remplacer nos véhicules diesel par des véhicul...,NaN,NaN
7,383f4915-42c7-4be6-a174-e996ccb2d761,ANTEA FRANCE,Entreprise,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,NaN,Opérationnel,3179.10,Plan d&#039;actions en cours de validation ( R...,NaN,https://www.anteagroup.fr/uploads/media/file/1...
8,ba2cc98d-b60c-4617-bcad-95a9510b7568,AC ENVIRONNEMENT,Entreprise,"Analyses, essais et inspections techniques",2023,Entre 250 et 499,NaN,Opérationnel,2624.00,Etudier les entreprises de services sous-trait...,NaN,NaN
9,16a01174-756d-4af2-a94b-4e817aae89b2,Eco CO2,Entreprise,"Activités spécialisées, scientifiques et techn...",2023,Entre 100 et 199,NaN,Opérationnel,1447.73,LibelléVolume de réductionIndicateurMoyen / Ob...,-18% des émissions à périmètre constant,NaN


In [46]:
target_firms.head(30)

,Id,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,Assujetti à la CSRD,Mode de consolidation,visible_indirect_emissions,Actions et moyens,Objectif de réduction pour 2030,Lien URL vers le rapport complet du BEGES
0,95a9d2c8-2be5-4d42-a5c0-4950a2a57ce8,SCOP EcoZimut,Entreprise,"Ingénierie, études techniques",2024,16,NaN,Opérationnel,6915.00,"Projets et usagesLâcher les gaz, pour plus de ...",NaN,NaN
1,7a0b6f91-9d03-4851-a2cc-9f2ab53b21d5,CARSO - LABORATOIRE SANTE ENVIRONNEMENT HYGIEN...,Entreprise,"Analyses, essais et inspections techniques",2024,890,NaN,Opérationnel,6453.00,Le groupe CARSO investit pour réduire ses émis...,42% de réduction des émissions,NaN
2,7abf0530-3f2c-46e1-8386-497989ef799c,SOCOTEC,Entreprise,"Analyses, essais et inspections techniques",2024,6075,oui,Opérationnel,4090.87,"Depuis 2022, SOCOTEC mets en œuvre un plan d’a...",42% de réduction sur le scope 1 et 2,NaN
3,d70a0f1c-b69c-4600-994b-80d1c903b8b5,ECOCERT SA,Entreprise,"Analyses, essais et inspections techniques",2024,778,NaN,Opérationnel,1242.70,La gouvernance a été renforcée avec l’évolutio...,NaN,NaN
4,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN
5,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN
6,5e1dcbd9-1848-4e05-a510-ef28436ed9ea,SAS OBJECTIF ZERO CARBONE CONSULTING (CARBONO),Entreprise,"Activités spécialisées, scientifiques et techn...",2024,6,NaN,Opérationnel,9.08,Remplacer nos véhicules diesel par des véhicul...,NaN,NaN
7,383f4915-42c7-4be6-a174-e996ccb2d761,ANTEA FRANCE,Entreprise,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,NaN,Opérationnel,3179.10,Plan d&#039;actions en cours de validation ( R...,NaN,https://www.anteagroup.fr/uploads/media/file/1...
8,ba2cc98d-b60c-4617-bcad-95a9510b7568,AC ENVIRONNEMENT,Entreprise,"Analyses, essais et inspections techniques",2023,Entre 250 et 499,NaN,Opérationnel,2624.00,Etudier les entreprises de services sous-trait...,NaN,NaN
9,16a01174-756d-4af2-a94b-4e817aae89b2,Eco CO2,Entreprise,"Activités spécialisées, scientifiques et techn...",2023,Entre 100 et 199,NaN,Opérationnel,1447.73,LibelléVolume de réductionIndicateurMoyen / Ob...,-18% des émissions à périmètre constant,NaN


## Inspect serious candidate firms

This block isolates the most relevant organizations from the first candidate search.

The goal is to compare firms that are large enough to have meaningful BEGES reporting, active in environmental or technical advisory fields, and likely to provide enough Scope 3 and action-plan information for downstream analysis.

In [48]:
serious_candidates = conn.sql("""
SELECT *
FROM target_firms
WHERE "Raison sociale" IN (
    'SOCOTEC',
    'ECOCERT SA',
    'Ecocert',
    'ANTEA FRANCE',
    'ARTELIA',
    'Eco CO2',
    'Envol Environnement'
)
ORDER BY "Raison sociale", "Année de reporting" DESC
""").df()

serious_candidates

,Id,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,Assujetti à la CSRD,Mode de consolidation,visible_indirect_emissions,Actions et moyens,Objectif de réduction pour 2030,Lien URL vers le rapport complet du BEGES
0,383f4915-42c7-4be6-a174-e996ccb2d761,ANTEA FRANCE,Entreprise,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,NaN,Opérationnel,3179.10,Plan d&#039;actions en cours de validation ( R...,NaN,https://www.anteagroup.fr/uploads/media/file/1...
1,160374fb-92ed-4227-ad2d-e5e0b6d60cc2,ARTELIA,Entreprise,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,NaN,Opérationnel,12938.00,MOBILITÉLes déplacements constituent notre pre...,Conformément à notre engagement auprès de l'in...,NaN
2,d70a0f1c-b69c-4600-994b-80d1c903b8b5,ECOCERT SA,Entreprise,"Analyses, essais et inspections techniques",2024,778,NaN,Opérationnel,1242.70,La gouvernance a été renforcée avec l’évolutio...,NaN,NaN
3,16a01174-756d-4af2-a94b-4e817aae89b2,Eco CO2,Entreprise,"Activités spécialisées, scientifiques et techn...",2023,Entre 100 et 199,NaN,Opérationnel,1447.73,LibelléVolume de réductionIndicateurMoyen / Ob...,-18% des émissions à périmètre constant,NaN
4,80308231-dd72-4b9f-bbbd-e04943f5b97f,Ecocert,Entreprise,"Analyses, essais et inspections techniques",2022,Entre 1 000 et 1 999,NaN,Opérationnel,2517.60,Scope 1Nous voulons diminuer l&#039;empreinte ...,NaN,Méthodologie BEGES v5
5,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN
6,63a4ef27-68c7-41bc-96cd-969faa5d140e,Envol Environnement,Entreprise,"Activités spécialisées, scientifiques et techn...",2024,100,non,Opérationnel,392.24,Limitation de la vitesse sur autoroute à 110 k...,Réduction de 100tCO2e,NaN


## Score candidate reports by data completeness

This block creates a reporting-quality score for candidate organizations.

The goal is to identify BEGES reports that are useful for a comparative analysis. Firms are scored based on whether they report indirect emissions, provide action-plan text, include reduction objectives, discuss uncertainty, list emission factors, and provide sources.

This avoids choosing firms based only on company name and makes the selection process more reproducible.

In [50]:
quality_candidates = conn.sql("""
SELECT
    "Id",
    "Raison sociale",
    "Type de structure",
    "APE(NAF) associé",
    "Libellé",
    "Année de reporting",
    "Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan" AS staff_count,
    "Mode de consolidation",

    (
        COALESCE("Emissions publication P3.1", 0) +
        COALESCE("Emissions publication P3.2", 0) +
        COALESCE("Emissions publication P3.3", 0) +
        COALESCE("Emissions publication P4.1", 0) +
        COALESCE("Emissions publication P4.2", 0) +
        COALESCE("Emissions publication P5.1", 0)
    ) AS visible_indirect_emissions,

    CASE WHEN "Actions et moyens" IS NOT NULL AND LENGTH("Actions et moyens") > 20 THEN 1 ELSE 0 END AS has_action_plan,
    CASE WHEN "Objectif de réduction pour 2030" IS NOT NULL AND LENGTH("Objectif de réduction pour 2030") > 5 THEN 1 ELSE 0 END AS has_2030_target,
    CASE WHEN "Liste des facteurs d’émission (FE) et Pouvoir de Réchauffement Global (PRG) utilisés" IS NOT NULL THEN 1 ELSE 0 END AS has_emission_factors,
    CASE WHEN "Incertitudes" IS NOT NULL THEN 1 ELSE 0 END AS has_uncertainty,
    CASE WHEN "Sources" IS NOT NULL THEN 1 ELSE 0 END AS has_sources,

    (
        CASE WHEN (
            COALESCE("Emissions publication P3.1", 0) +
            COALESCE("Emissions publication P3.2", 0) +
            COALESCE("Emissions publication P3.3", 0) +
            COALESCE("Emissions publication P4.1", 0) +
            COALESCE("Emissions publication P4.2", 0) +
            COALESCE("Emissions publication P5.1", 0)
        ) > 0 THEN 2 ELSE 0 END +
        CASE WHEN "Actions et moyens" IS NOT NULL AND LENGTH("Actions et moyens") > 20 THEN 2 ELSE 0 END +
        CASE WHEN "Objectif de réduction pour 2030" IS NOT NULL AND LENGTH("Objectif de réduction pour 2030") > 5 THEN 1 ELSE 0 END +
        CASE WHEN "Liste des facteurs d’émission (FE) et Pouvoir de Réchauffement Global (PRG) utilisés" IS NOT NULL THEN 1 ELSE 0 END +
        CASE WHEN "Incertitudes" IS NOT NULL THEN 1 ELSE 0 END +
        CASE WHEN "Sources" IS NOT NULL THEN 1 ELSE 0 END
    ) AS report_quality_score,

    "Lien URL vers le rapport complet du BEGES"

FROM beges
WHERE "Année de reporting" >= 2022
  AND "Libellé" IN (
    'Ingénierie, études techniques',
    'Conseil pour les affaires et autres conseils de gestion',
    'Analyses, essais et inspections techniques',
    'Activités spécialisées, scientifiques et techniques diverses'
  )
ORDER BY report_quality_score DESC, visible_indirect_emissions DESC
""").df()

quality_candidates.head(50)

,Id,Raison sociale,Type de structure,APE(NAF) associé,Libellé,Année de reporting,staff_count,Mode de consolidation,visible_indirect_emissions,has_action_plan,has_2030_target,has_emission_factors,has_uncertainty,has_sources,report_quality_score,Lien URL vers le rapport complet du BEGES
0,0dd8ffd7-5b30-405c-a68a-50a7b605502c,EXCENT FRANCE,Entreprise,7112B,"Ingénierie, études techniques",2023,Entre 500 et 999,Opérationnel,6867080.000,1,1,1,1,1,8,NaN
1,7dd0825b-dc24-4a94-99c8-53f00026fb3d,EXCENT FRANCE,Entreprise,7112B,"Ingénierie, études techniques",2022,Entre 500 et 999,Opérationnel,6771822.000,1,1,1,1,1,8,NaN
2,4fb3a4f9-e615-4ac3-952d-743a4bdf3fd3,Clariane,Entreprise,7022Z,Conseil pour les affaires et autres conseils d...,2023,60000,Opérationnel,489788.700,1,1,1,1,1,8,Le BEGES 2023 a été publié dans l'URD 2024 du ...
3,1d9acfee-ef13-4fdd-971f-245bde451611,Clariane,Entreprise,7022Z,Conseil pour les affaires et autres conseils d...,2024,63086,Opérationnel,442526.000,1,1,1,1,1,8,"Publication au sein de l'URD 2024 du Groupe, e..."
4,424b82d5-3e48-4cc0-b52f-ad3d00b813b6,CGX AERO,Entreprise,7112B,"Ingénierie, études techniques",2023,70,Opérationnel,388197.000,1,1,1,1,1,8,NaN
5,e158bbd3-b40d-4dc4-9071-7bb9d391dad0,BEG INGENIERIE,Entreprise,7112B,"Ingénierie, études techniques",2023,Entre 100 et 199,Opérationnel,154684.000,1,1,1,1,1,8,NaN
6,4f00070e-408e-4a86-a519-59c1bb6be503,EPSA,Entreprise,7022Z,Conseil pour les affaires et autres conseils d...,2024,2274,Opérationnel,26724.000,1,1,1,1,1,8,NaN
7,2e265c3e-1ed3-4a64-a1f8-9d3b2937b72a,REFLEX CES,Entreprise,7112B,"Ingénierie, études techniques",2022,Entre 50 et 99,Opérationnel,13035.000,1,1,1,1,1,8,NaN
8,160374fb-92ed-4227-ad2d-e5e0b6d60cc2,ARTELIA,Entreprise,7112B,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,Opérationnel,12938.000,1,1,1,1,1,8,NaN
9,5d3d794b-a3f8-4e63-85ef-578ed446de4a,GINGER CEBTP,Entreprise,7112B,"Ingénierie, études techniques",2022,Entre 1 000 et 1 999,Opérationnel,12926.000,1,1,1,1,1,8,NaN


In [51]:
quality_candidates[
    [
        "Raison sociale",
        "Type de structure",
        "Libellé",
        "Année de reporting",
        "staff_count",
        "visible_indirect_emissions",
        "has_action_plan",
        "has_2030_target",
        "has_emission_factors",
        "has_uncertainty",
        "has_sources",
        "report_quality_score"
    ]
].head(49)

,Raison sociale,Type de structure,Libellé,Année de reporting,staff_count,visible_indirect_emissions,has_action_plan,has_2030_target,has_emission_factors,has_uncertainty,has_sources,report_quality_score
0,EXCENT FRANCE,Entreprise,"Ingénierie, études techniques",2023,Entre 500 et 999,6867080.000,1,1,1,1,1,8
1,EXCENT FRANCE,Entreprise,"Ingénierie, études techniques",2022,Entre 500 et 999,6771822.000,1,1,1,1,1,8
2,Clariane,Entreprise,Conseil pour les affaires et autres conseils d...,2023,60000,489788.700,1,1,1,1,1,8
3,Clariane,Entreprise,Conseil pour les affaires et autres conseils d...,2024,63086,442526.000,1,1,1,1,1,8
4,CGX AERO,Entreprise,"Ingénierie, études techniques",2023,70,388197.000,1,1,1,1,1,8
5,BEG INGENIERIE,Entreprise,"Ingénierie, études techniques",2023,Entre 100 et 199,154684.000,1,1,1,1,1,8
6,EPSA,Entreprise,Conseil pour les affaires et autres conseils d...,2024,2274,26724.000,1,1,1,1,1,8
7,REFLEX CES,Entreprise,"Ingénierie, études techniques",2022,Entre 50 et 99,13035.000,1,1,1,1,1,8
8,ARTELIA,Entreprise,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,12938.000,1,1,1,1,1,8
9,GINGER CEBTP,Entreprise,"Ingénierie, études techniques",2022,Entre 1 000 et 1 999,12926.000,1,1,1,1,1,8


## Inspect organization descriptions

This block reviews the organization descriptions for high-quality candidate reports.

The purpose is to understand what each organization actually does before selecting the final three firms. This prevents the project from choosing firms only because their BEGES reports are complete, while ignoring whether their activities are relevant to environmental, sustainability, carbon, or technical advisory work.

In [53]:
conn.register("quality_candidates", quality_candidates)

org_descriptions = conn.sql("""
SELECT
    q."Raison sociale",
    q."Libellé",
    q."Année de reporting",
    q.staff_count,
    q.visible_indirect_emissions,
    q.report_quality_score,
    b."Présentation de l'organisation"
FROM quality_candidates q
LEFT JOIN beges b
    ON q."Id" = b."Id"
WHERE q."Raison sociale" IN (
    'ARTELIA',
    'GINGER CEBTP',
    'BUREAU VERITAS EXPLOITATION',
    'CARSO - LABORATOIRE SANTE ENVIRONNEMENT HYGIENE DE LYON',
    'Eco CO2',
    'SYSTRA France',
    'SOCOTEC',
    'ANTEA FRANCE',
    'ECOCERT SA'
)
ORDER BY q.report_quality_score DESC, q.visible_indirect_emissions DESC
""").df()

org_descriptions


,Raison sociale,Libellé,Année de reporting,staff_count,visible_indirect_emissions,report_quality_score,Présentation de l'organisation
0,ARTELIA,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,12938.00,8,"Artelia, une ingénierie indépendante &amp; mul..."
1,GINGER CEBTP,"Ingénierie, études techniques",2022,Entre 1 000 et 1 999,12926.00,8,GINGER CEBTP est un groupe spécialisé dans l’i...
2,BUREAU VERITAS EXPLOITATION,"Analyses, essais et inspections techniques",2023,Entre 2 000 et 4 999,6560.00,8,En tant qu’entreprise de services « Business t...
3,CARSO - LABORATOIRE SANTE ENVIRONNEMENT HYGIEN...,"Analyses, essais et inspections techniques",2024,890,6453.00,8,"CARSO LSEHL, basé à Vénissieux (69), est un de..."
4,SYSTRA France,"Ingénierie, études techniques",2023,Entre 2 000 et 4 999,1638.00,8,SYSTRA est l’un des leaders mondiaux d’ingénie...
5,Eco CO2,"Activités spécialisées, scientifiques et techn...",2023,Entre 100 et 199,1447.73,8,Eco CO2 accompagne les entreprises et les coll...
6,SYSTRA France,"Ingénierie, études techniques",2024,2073,1298.00,8,SYSTRA est l’un des premiers groupes mondiaux ...
7,ECOCERT SA,"Analyses, essais et inspections techniques",2024,778,1242.70,7,"Depuis plus de 30 ans, nous accompagnons de no..."
8,ANTEA FRANCE,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,3179.10,6,Nos équipes composées de 1 000 collaborateurs ...
9,SYSTRA France,"Ingénierie, études techniques",2022,Entre 2 000 et 4 999,1695.00,5,SYSTRA est l’un des leaders mondiaux d’ingénie...


In [54]:
pd.set_option("display.max_colwidth", 500)

org_descriptions[
    [
        "Raison sociale",
        "Libellé",
        "Année de reporting",
        "staff_count",
        "visible_indirect_emissions",
        "Présentation de l'organisation"
    ]
]

,Raison sociale,Libellé,Année de reporting,staff_count,visible_indirect_emissions,Présentation de l'organisation
0,ARTELIA,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,12938.00,"Artelia, une ingénierie indépendante &amp; multidisciplinaireUn accompagnement sur l’ensemble du cycle de vie de votre projetAssociant de fortes compétences en ingénierie de la mobilité, de l’eau, de l’énergie, du bâtiment et de l’industrie, Artelia offre une gamme complète de services, de l’expertise à la livraison de projets complexes : consulting, études et schémas directeurs, management de projet, maîtrise d’œuvre de conception et d’exécution, gestion patrimoniale et marchés globaux. Ave..."
1,GINGER CEBTP,"Ingénierie, études techniques",2022,Entre 1 000 et 1 999,12926.00,"GINGER CEBTP est un groupe spécialisé dans l’ingénierie des sols, des matériaux et des ouvrages. Il intervient en tant que bureau d’étude auprès de tous les acteurs de la construction dans l’accompagnement des projets des plus simples aux plus complexes. Tout au long du cycle de vie d’un ouvrage quel qu’il soit, GINGER CEBTP offre une prise en charge globale des missions d’expertises en se positionnant sur une chaîne de valeur unique :"
2,BUREAU VERITAS EXPLOITATION,"Analyses, essais et inspections techniques",2023,Entre 2 000 et 4 999,6560.00,"En tant qu’entreprise de services « Business to Business to Society », nous établissons une relation de confiance entre entreprises, pouvoirs publics et consommateurs. Nous répondons, à travers cette mission, aux défis sociétaux et environnementaux majeurs de notre époque. Notre mission consiste à réduire les risques de nos clients, à améliorer leurs performances et à soutenir leurs efforts d&#039;innovation. Et ce, en tenant compte de leurs obligations normatives, en matière de qualité, de ..."
3,CARSO - LABORATOIRE SANTE ENVIRONNEMENT HYGIENE DE LYON,"Analyses, essais et inspections techniques",2024,890,6453.00,"CARSO LSEHL, basé à Vénissieux (69), est un des acteurs clef Français de l’analyse environnementale, pharmaceutique et agroalimentaire. L’entreprise, forte de ses 900 collaborateurs, est organisée autour de 4 grands blocs : les laboratoires, les prélèvements, le support à la production et les services généraux. Les équipes de prélèvements jouent un rôle crucial, réalisant une grande partie des prélèvements en environnement et en hygiène hospitalière. Ces opérations , souvent techniques et co..."
4,SYSTRA France,"Ingénierie, études techniques",2023,Entre 2 000 et 4 999,1638.00,"SYSTRA est l’un des leaders mondiaux d’ingénierie et de conseil spécialisés dans les transports publics et les solutions de mobilité. Depuis plus de 60 ans, le Groupe s’engage auprès des villes et des territoires pour contribuer à leur développement en créant, améliorant et modernisant leurs infrastructures et systèmes de transport.En tant qu&#039;acteur de la transition énergétique, SYSTRA propose à ses clients des solutions et des services couvrant l&#039;ensemble du cycle de vie des proje..."
5,Eco CO2,"Activités spécialisées, scientifiques et techniques diverses",2023,Entre 100 et 199,1447.73,"Eco CO2 accompagne les entreprises et les collectivités dans leurs pratiques environnementales, sociales et de gouvernance pour construire un nouveau modèle économique performant et responsable.Notre expertise en mobilité et transportAux côtés de l’ADEME et des organisations professionnelles du secteur, Eco CO2 aide les entreprises et les collectivités dans la réduction de leur impact environnemental, la transition vers la logistique durable, le report modal, et le développement des mobilité..."
6,SYSTRA France,"Ingénierie, études techniques",2024,2073,1298.00,"SYSTRA est l’un des premiers groupes mondiaux d’ingénierie et de conseil spécialisés dans les transports publics et les solutions de mobilité. Depuis plus de 65 ans, le Groupe s’engage auprès des villes et des territoires pour contribuer à leur développement en créant, améliorant et modernisant

## Select final firms for detailed analysis

This block defines the final firm set for the project.

The selected organizations represent five related but distinct environmental and sustainability archetypes: environmental engineering, technical consulting, compliance and inspection, sustainability certification, and ecological transition advisory.

This selection allows the project to compare not only emissions totals, but also differences in reporting quality, Scope 3 coverage, methodological transparency, and transition-plan detail across organizations whose public missions are connected to environmental performance.

In [55]:
selected_firms = org_descriptions[
    org_descriptions["Raison sociale"].isin([
        "ARTELIA",
        "ANTEA FRANCE",
        "BUREAU VERITAS EXPLOITATION",
        "ECOCERT SA",
        "Eco CO2"
    ])
].copy()

selected_firms

,Raison sociale,Libellé,Année de reporting,staff_count,visible_indirect_emissions,report_quality_score,Présentation de l'organisation
0,ARTELIA,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,12938.00,8,"Artelia, une ingénierie indépendante &amp; multidisciplinaireUn accompagnement sur l’ensemble du cycle de vie de votre projetAssociant de fortes compétences en ingénierie de la mobilité, de l’eau, de l’énergie, du bâtiment et de l’industrie, Artelia offre une gamme complète de services, de l’expertise à la livraison de projets complexes : consulting, études et schémas directeurs, management de projet, maîtrise d’œuvre de conception et d’exécution, gestion patrimoniale et marchés globaux. Ave..."
2,BUREAU VERITAS EXPLOITATION,"Analyses, essais et inspections techniques",2023,Entre 2 000 et 4 999,6560.00,8,"En tant qu’entreprise de services « Business to Business to Society », nous établissons une relation de confiance entre entreprises, pouvoirs publics et consommateurs. Nous répondons, à travers cette mission, aux défis sociétaux et environnementaux majeurs de notre époque. Notre mission consiste à réduire les risques de nos clients, à améliorer leurs performances et à soutenir leurs efforts d&#039;innovation. Et ce, en tenant compte de leurs obligations normatives, en matière de qualité, de ..."
5,Eco CO2,"Activités spécialisées, scientifiques et techniques diverses",2023,Entre 100 et 199,1447.73,8,"Eco CO2 accompagne les entreprises et les collectivités dans leurs pratiques environnementales, sociales et de gouvernance pour construire un nouveau modèle économique performant et responsable.Notre expertise en mobilité et transportAux côtés de l’ADEME et des organisations professionnelles du secteur, Eco CO2 aide les entreprises et les collectivités dans la réduction de leur impact environnemental, la transition vers la logistique durable, le report modal, et le développement des mobilité..."
7,ECOCERT SA,"Analyses, essais et inspections techniques",2024,778,1242.70,7,"Depuis plus de 30 ans, nous accompagnons de nombreux acteurs dans le déploiement et la valorisation de pratiques durables à travers la certification, le conseil et la formation. Engagé dès sa création pour l’agriculture biologique, Ecocert a aujourd&#039;hui étendu son action à de nombreuses filières dans le monde Notre mission : Agir pour un monde durable en accompagnant la transition vers des modèles économiques plus juste et qui préservent la santé, le climat et la biodiversité à travers ..."
8,ANTEA FRANCE,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,3179.10,6,"Nos équipes composées de 1 000 collaborateurs et réparties dans 25 implantations en métropole et 5 en Départements d’Outre-Mer, interviennent au cœur des territoires et aux côtés des acteurs locaux en France métropolitaine, dans les DOM et à l’international.Nous sommes présents auprès d’une clientèle très diversifiée, constituée de grands groupes industriels, de PME dynamiques, de collectivités locales ou de gouvernements. A tous, nous avons à cœur de répondre avec le même engagement et la m..."


In [56]:
selected_firms = org_descriptions[
    org_descriptions["Raison sociale"].isin([
        "ARTELIA",
        "ANTEA FRANCE",
        "BUREAU VERITAS EXPLOITATION",
        "ECOCERT SA",
        "Eco CO2"
    ])
].copy()

selected_firms = (
    selected_firms
    .sort_values(["Raison sociale", "Année de reporting"], ascending=[True, False])
    .drop_duplicates(subset=["Raison sociale"], keep="first")
)

selected_firms

,Raison sociale,Libellé,Année de reporting,staff_count,visible_indirect_emissions,report_quality_score,Présentation de l'organisation
8,ANTEA FRANCE,"Ingénierie, études techniques",2023,Entre 1 000 et 1 999,3179.10,6,"Nos équipes composées de 1 000 collaborateurs et réparties dans 25 implantations en métropole et 5 en Départements d’Outre-Mer, interviennent au cœur des territoires et aux côtés des acteurs locaux en France métropolitaine, dans les DOM et à l’international.Nous sommes présents auprès d’une clientèle très diversifiée, constituée de grands groupes industriels, de PME dynamiques, de collectivités locales ou de gouvernements. A tous, nous avons à cœur de répondre avec le même engagement et la m..."
0,ARTELIA,"Ingénierie, études techniques",2022,Entre 5 000 et 9 999,12938.00,8,"Artelia, une ingénierie indépendante &amp; multidisciplinaireUn accompagnement sur l’ensemble du cycle de vie de votre projetAssociant de fortes compétences en ingénierie de la mobilité, de l’eau, de l’énergie, du bâtiment et de l’industrie, Artelia offre une gamme complète de services, de l’expertise à la livraison de projets complexes : consulting, études et schémas directeurs, management de projet, maîtrise d’œuvre de conception et d’exécution, gestion patrimoniale et marchés globaux. Ave..."
2,BUREAU VERITAS EXPLOITATION,"Analyses, essais et inspections techniques",2023,Entre 2 000 et 4 999,6560.00,8,"En tant qu’entreprise de services « Business to Business to Society », nous établissons une relation de confiance entre entreprises, pouvoirs publics et consommateurs. Nous répondons, à travers cette mission, aux défis sociétaux et environnementaux majeurs de notre époque. Notre mission consiste à réduire les risques de nos clients, à améliorer leurs performances et à soutenir leurs efforts d&#039;innovation. Et ce, en tenant compte de leurs obligations normatives, en matière de qualité, de ..."
7,ECOCERT SA,"Analyses, essais et inspections techniques",2024,778,1242.70,7,"Depuis plus de 30 ans, nous accompagnons de nombreux acteurs dans le déploiement et la valorisation de pratiques durables à travers la certification, le conseil et la formation. Engagé dès sa création pour l’agriculture biologique, Ecocert a aujourd&#039;hui étendu son action à de nombreuses filières dans le monde Notre mission : Agir pour un monde durable en accompagnant la transition vers des modèles économiques plus juste et qui préservent la santé, le climat et la biodiversité à travers ..."
5,Eco CO2,"Activités spécialisées, scientifiques et techniques diverses",2023,Entre 100 et 199,1447.73,8,"Eco CO2 accompagne les entreprises et les collectivités dans leurs pratiques environnementales, sociales et de gouvernance pour construire un nouveau modèle économique performant et responsable.Notre expertise en mobilité et transportAux côtés de l’ADEME et des organisations professionnelles du secteur, Eco CO2 aide les entreprises et les collectivités dans la réduction de leur impact environnemental, la transition vers la logistique durable, le report modal, et le développement des mobilité..."


## Save selected firm list

This block saves the final selected firms as a processed CSV file.

This creates a reproducible checkpoint for the project. Downstream notebooks can load this selected firm list directly instead of repeating the firm-selection logic.

In [58]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

selected_firms.to_csv(processed_dir / "selected_firms.csv", index=False)

print("Saved selected_firms.csv")

Saved selected_firms.csv


## Firm selection summary

The final selected firms are:

ARTELIA

ANTEA FRANCE

BUREAU VERITAS EXPLOITATION

ECOCERT SA

Eco CO2

These firms were selected because they represent complementary environmental and sustainability-related organization types:

ARTELIA and ANTEA FRANCE represent environmental and multidisciplinary engineering consulting.

BUREAU VERITAS EXPLOITATION represents inspection, compliance, risk management, and certification.

ECOCERT SA represents sustainability certification, consulting, and training.

Eco CO2 represents ecological transition, ESG practices, mobility, and climate advisory.

Together, these organizations provide a strong basis for comparing greenhouse gas inventory structure, Scope 3 coverage, methodological transparency, and transition-plan quality across firms whose public missions are connected to environmental performance.

In [59]:
selected_ids = selected_firms["Raison sociale"].tolist()

scope_check = conn.sql("""
SELECT
    "Raison sociale",
    "Année de reporting",

    COALESCE("Emissions publication P3.1", 0) AS p3_1,
    COALESCE("Emissions publication P3.2", 0) AS p3_2,
    COALESCE("Emissions publication P3.3", 0) AS p3_3,
    COALESCE("Emissions publication P3.4", 0) AS p3_4,
    COALESCE("Emissions publication P3.5", 0) AS p3_5,

    COALESCE("Emissions publication P4.1", 0) AS p4_1,
    COALESCE("Emissions publication P4.2", 0) AS p4_2,
    COALESCE("Emissions publication P4.3", 0) AS p4_3,
    COALESCE("Emissions publication P4.4", 0) AS p4_4,
    COALESCE("Emissions publication P4.5", 0) AS p4_5,

    COALESCE("Emissions publication P5.1", 0) AS p5_1,
    COALESCE("Emissions publication P5.2", 0) AS p5_2,
    COALESCE("Emissions publication P5.3", 0) AS p5_3,
    COALESCE("Emissions publication P5.4", 0) AS p5_4,

    (
        COALESCE("Emissions publication P3.1", 0) +
        COALESCE("Emissions publication P3.2", 0) +
        COALESCE("Emissions publication P3.3", 0) +
        COALESCE("Emissions publication P3.4", 0) +
        COALESCE("Emissions publication P3.5", 0) +
        COALESCE("Emissions publication P4.1", 0) +
        COALESCE("Emissions publication P4.2", 0) +
        COALESCE("Emissions publication P4.3", 0) +
        COALESCE("Emissions publication P4.4", 0) +
        COALESCE("Emissions publication P4.5", 0) +
        COALESCE("Emissions publication P5.1", 0) +
        COALESCE("Emissions publication P5.2", 0) +
        COALESCE("Emissions publication P5.3", 0) +
        COALESCE("Emissions publication P5.4", 0)
    ) AS total_indirect_emissions

FROM beges
WHERE "Raison sociale" IN (
    'ARTELIA',
    'ANTEA FRANCE',
    'BUREAU VERITAS EXPLOITATION',
    'ECOCERT SA',
    'Eco CO2'
)
ORDER BY "Raison sociale", "Année de reporting" DESC
""").df()

scope_check

,Raison sociale,Année de reporting,p3_1,p3_2,p3_3,p3_4,p3_5,p4_1,p4_2,p4_3,p4_4,p4_5,p5_1,p5_2,p5_3,p5_4,total_indirect_emissions
0,ANTEA FRANCE,2023,11.50,0.0,552.90,0.0,119.60,1283.30,1331.40,63.70,1315.5,8162.90,0.00,0.0,0.00,0.0,12840.80
1,ANTEA FRANCE,2019,0.00,24.0,743.00,0.0,280.00,7652.00,1045.00,0.00,0.0,0.00,0.00,0.0,0.00,0.0,9744.00
2,ARTELIA,2022,0.00,0.0,5925.00,0.0,4968.00,1966.00,5047.00,38.00,0.0,0.00,0.00,0.0,0.00,0.0,17944.00
3,BUREAU VERITAS EXPLOITATION,2023,0.00,0.0,1253.00,0.0,0.00,4912.00,395.00,62.00,0.0,16446.00,0.00,0.0,0.00,0.0,23068.00
4,BUREAU VERITAS EXPLOITATION,2019,0.00,0.0,0.00,0.0,1455.00,2.00,0.00,62.00,0.0,0.00,0.00,0.0,0.00,0.0,1519.00
5,ECOCERT SA,2024,90.40,0.0,460.50,10.4,551.60,362.00,329.80,0.80,396.4,981.70,0.00,0.0,0.00,0.0,3183.60
6,Eco CO2,2023,0.15,0.0,15.46,0.0,61.85,1353.04,78.63,0.23,0.0,3.48,0.45,0.0,0.63,0.0,1513.92


## Verify indirect emissions coverage

This block verifies that all selected firms report indirect emissions categories in the ADEME BEGES dataset.

This step is necessary because the project depends on comparing Scope 3-type reporting. In the French BEGES structure, the relevant data are reported through indirect emissions categories such as P3, P4, and P5. These categories are not identical to the GHG Protocol Scope 3 categories, but they provide the correct basis for analyzing indirect emissions coverage within the BEGES framework.

In [60]:
scope_check.to_csv(processed_dir / "selected_firms_indirect_emissions_check.csv", index=False)

print("Saved selected_firms_indirect_emissions_check.csv")

Saved selected_firms_indirect_emissions_check.csv
